# 3.2.2 — Projeção de Embeddings de **Token** com BERT

**Aluna:** Bruna Soto Cardoso dos Santos  
**Corpus:** 35 documentos — Ciência e Tecnologia no Brasil (AD1)  
**Repositório:** https://github.com/brusoto/SRI_AD_TE

---
Gera embeddings BERT contextuais para cada token do corpus e salva arquivos `.tsv` para o [TensorFlow Embedding Projector](https://projector.tensorflow.org/).

**Saídas** (salvas em `projecao/`):
- `tensors_token.tsv` — vetores de embedding de tokens (768 dim)
- `metadata_token.tsv` — rótulos dos tokens (palavra, doc, contexto)
- `config_token.json` — configuração para o Projector

## Célula 1 — Instalação de dependências

In [ ]:
!pip install transformers torch --quiet

## Célula 2 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/SRI'
DATA_PATH = os.path.join(BASE_PATH, 'data')
PROJ_PATH = os.path.join(BASE_PATH, 'projecao')
os.makedirs(PROJ_PATH, exist_ok=True)

CSV_FILE = os.path.join(DATA_PATH, 'documentos.csv')
print(f'CSV: {CSV_FILE}')

## Célula 3 — Baixar corpus se necessário

In [ ]:
if not os.path.exists(CSV_FILE):
    os.makedirs(DATA_PATH, exist_ok=True)
    URL = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/data/documentos.csv'
    !wget -q "{URL}" -O "{CSV_FILE}"
    print('documentos.csv baixado.')
else:
    print('documentos.csv já existe.')

## Célula 4 — Imports e carregamento do corpus

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertModel

df = pd.read_csv(CSV_FILE)
df.columns = [c.strip().lower() for c in df.columns]
if 'id' not in df.columns:
    df.insert(0, 'id', range(1, len(df)+1))

print(f'Documentos: {len(df)}')
df.head(2)

## Célula 5 — Carregamento do modelo BERT

In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)
model      = BertModel.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f'Dispositivo: {device}')

## Célula 6 — Parâmetros de filtragem de tokens

> **MAX_TOKENS_POR_DOC**: limita quantos tokens por documento são projetados.  
> Com 35 docs × 30 tokens = ~1050 pontos no Projector (valor recomendado).

In [ ]:
# ── Ajuste aqui se quiser mais ou menos tokens no gráfico ────────────────────
MAX_TOKENS_POR_DOC = 30    # tokens por documento (exclui [CLS] e [SEP])
EXCLUIR_SUBWORDS   = True  # True = ignora tokens que começam com ##
EXCLUIR_PONTUACAO  = True  # True = ignora tokens que são só pontuação

import string
PONTUACOES = set(string.punctuation)

## Célula 7 — Geração dos embeddings de token

In [ ]:
token_embeddings = []
token_labels     = []
token_metadata   = []  # (token, doc_id, contexto)

for _, row in df.iterrows():
    doc_id = row['id']
    texto  = str(row['documento'])

    # Tokeniza
    encoded = tokenizer(
        texto,
        return_tensors='pt',
        max_length=512,
        truncation=True,
        padding='max_length'
    )
    encoded_device = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded_device)

    # hidden states: (1, seq_len, 768)
    hidden = outputs.last_hidden_state.squeeze(0).cpu().numpy()

    # Recupera lista de tokens
    ids    = encoded['input_ids'].squeeze().tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)

    count = 0
    for idx, (tok, emb) in enumerate(zip(tokens, hidden)):
        # Pula tokens especiais
        if tok in ('[CLS]', '[SEP]', '[PAD]'):
            continue
        if EXCLUIR_SUBWORDS and tok.startswith('##'):
            continue
        if EXCLUIR_PONTUACAO and tok in PONTUACOES:
            continue
        if count >= MAX_TOKENS_POR_DOC:
            break

        # Contexto = 3 tokens ao redor
        ctx_start = max(1, idx-2)
        ctx_end   = min(len(tokens)-1, idx+3)
        contexto  = ' '.join(t for t in tokens[ctx_start:ctx_end]
                             if t not in ('[CLS]','[SEP]','[PAD]'))

        token_embeddings.append(emb)
        token_labels.append(f'{tok} (Doc_{doc_id})')
        token_metadata.append((tok, f'Doc_{doc_id}', contexto))
        count += 1

    print(f'Doc_{doc_id:02d}: {count} tokens extraídos')

token_embeddings_np = np.array(token_embeddings)
print(f'\nTotal tokens: {len(token_labels)}')
print(f'Shape       : {token_embeddings_np.shape}')

## Célula 8 — Salvar arquivos TSV

In [ ]:
# tensors_token.tsv
tensor_path = os.path.join(PROJ_PATH, 'tensors_token.tsv')
np.savetxt(tensor_path, token_embeddings_np, delimiter='\t')
print(f'Salvo: {tensor_path}')

# metadata_token.tsv
meta_path = os.path.join(PROJ_PATH, 'metadata_token.tsv')
with open(meta_path, 'w', encoding='utf-8') as f:
    f.write('token\tdoc\tcontexto\n')
    for tok, doc, ctx in token_metadata:
        f.write(f'{tok}\t{doc}\t{ctx}\n')
print(f'Salvo: {meta_path}')

## Célula 9 — Gerar config_token.json

In [ ]:
import json

GITHUB_RAW = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao'

config = {
    "embeddings": [
        {
            "tensorName": "Tokens — Ciência e Tecnologia BR",
            "tensorShape": [len(token_labels), 768],
            "tensorPath": f"{GITHUB_RAW}/tensors_token.tsv",
            "metadataPath": f"{GITHUB_RAW}/metadata_token.tsv"
        }
    ]
}

config_path = os.path.join(PROJ_PATH, 'config_token.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f'Salvo: {config_path}')

## Célula 10 — Visualização rápida PCA (opcional)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.cm as cm

pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(token_embeddings_np)

# Cores por documento
doc_ids_num = [int(m[1].replace('Doc_','')) for m in token_metadata]
n_docs      = df['id'].max()
cmap        = cm.get_cmap('tab20', n_docs)

fig, ax = plt.subplots(figsize=(15, 10))

for i, (x, y) in enumerate(coords):
    color = cmap(doc_ids_num[i] - 1)
    ax.scatter(x, y, color=color, s=20, alpha=0.7, zorder=3)

    # Anota apenas tokens de interesse (frequência alta ou tokens-chave)
    tok = token_metadata[i][0]
    if len(tok) > 3 and i % 15 == 0:  # amostra para não poluir
        ax.annotate(tok, (x, y), fontsize=6, alpha=0.8,
                    xytext=(2, 2), textcoords='offset points')

ax.set_title('Projeção PCA dos Embeddings de Token (BERT)\nCorpus: Ciência e Tecnologia no Brasil',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(PROJ_PATH, 'grafico_token.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {fig_path}')

## Célula 11 — Link para o TensorFlow Embedding Projector

In [ ]:
PROJECTOR_URL = (
    'https://projector.tensorflow.org/?config='
    'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao/config_token.json'
)
print('═' * 70)
print('LINK DO EMBEDDING PROJECTOR — TOKENS:')
print(PROJECTOR_URL)
print('═' * 70)